# 10. Data Quality Assessment and Cleaning

In this section, the cleaned relational tables exported from SQL are assessed for quality, consistency, and analytical readiness before moving into exploratory data analysis and feature engineering. The objective is to verify that the dataset is complete, structurally reliable, and suitable for churn analysis, retention segmentation, and business interpretation.

The assessment focuses on five main quality dimensions:

- completeness
- uniqueness
- referential integrity
- data type consistency
- value validity

Because the dataset is relational and multi-table by design, maintaining data quality at both the table level and the relationship level is essential to avoid misleading results in later stages of the analysis.

In [71]:
import pandas as pd
import numpy as np

In [72]:
# Update these file paths according to your folder structure
accounts = pd.read_csv("typed_accounts.csv")
subscriptions = pd.read_csv("typed_subscription.csv")
feature_usage = pd.read_csv("typed_feature_usage.csv")
support_tickets = pd.read_csv("typed_support_tickets.csv")
churn_events = pd.read_csv("typed_churn_events.csv")

## 10.2 Initial Structural Review

Before performing detailed quality checks, the dimensions and column structure of each table are reviewed to confirm that the SQL outputs were exported correctly and that the expected fields are available for analysis.

In [73]:
tables = {
    "accounts": accounts,
    "subscriptions": subscriptions,
    "feature_usage": feature_usage,
    "support_tickets": support_tickets,
    "churn_events": churn_events
}

for name, df in tables.items():
    print(f"\n{name.upper()}")
    print("Shape:", df.shape)
    print("Columns:", list(df.columns))


ACCOUNTS
Shape: (500, 10)
Columns: ['account_id', 'account_name', 'industry', 'country', 'signup_date', 'referral_source', 'plan_tier', 'seats', 'is_trial', 'churn_flag']

SUBSCRIPTIONS
Shape: (5000, 14)
Columns: ['subscription_id', 'account_id', 'start_date', 'end_date', 'plan_tier', 'seats', 'mrr_amount', 'arr_amount', 'is_trial', 'upgrade_flag', 'downgrade_flag', 'churn_flag', 'billing_frequency', 'auto_renew_flag']

FEATURE_USAGE
Shape: (25000, 8)
Columns: ['usage_id', 'subscription_id', 'usage_date', 'feature_name', 'usage_count', 'usage_duration_secs', 'error_count', 'is_beta_feature']

SUPPORT_TICKETS
Shape: (2000, 9)
Columns: ['ticket_id', 'account_id', 'submitted_at', 'closed_at', 'resolution_time_hours', 'priority', 'first_response_time_minutes', 'satisfaction_score', 'escalation_flag']

CHURN_EVENTS
Shape: (600, 9)
Columns: ['churn_event_id', 'account_id', 'churn_date', 'reason_code', 'refund_amount_usd', 'preceding_upgrade_flag', 'preceding_downgrade_flag', 'is_reactivatio

## 10.3 Missing Values Assessment

The next step is to evaluate completeness across all tables. Missing values are important because they can affect aggregation logic, segmentation, and churn interpretation. Some missing values may be acceptable depending on business context, but they must be identified explicitly before further analysis.

In [74]:
for name, df in tables.items():
    print(f"\n{name.upper()} - Missing Values")
    
    # تحويل القيم الفاضية لـ NaN
    df_clean = df.replace(["", " ", "NULL", "NaN"], None)
    
    # حساب missing الحقيقي
    missing = df_clean.isnull().sum().sort_values(ascending=False)
    
    # إظهار الأعمدة اللي فيها missing فقط
    if missing.sum() > 0:
        print(missing[missing > 0])
    else:
        print("No missing values found.")



ACCOUNTS - Missing Values
No missing values found.

SUBSCRIPTIONS - Missing Values
end_date    4514
dtype: int64

FEATURE_USAGE - Missing Values
No missing values found.

SUPPORT_TICKETS - Missing Values
No missing values found.

CHURN_EVENTS - Missing Values
feedback_text    148
dtype: int64


## 10.4 Duplicate Records Assessment

Duplicate records can distort churn metrics, inflate usage counts, and produce misleading business insights. For this reason, the dataset is checked for duplicates at both the full-row level and the primary-key level.

In [75]:
for name, df in tables.items():
    duplicate_rows = df.duplicated().sum()
    print(f"{name.upper()} - Full row duplicates: {duplicate_rows}")

ACCOUNTS - Full row duplicates: 0
SUBSCRIPTIONS - Full row duplicates: 0
FEATURE_USAGE - Full row duplicates: 0
SUPPORT_TICKETS - Full row duplicates: 0
CHURN_EVENTS - Full row duplicates: 0


In [76]:
print("ACCOUNT ID duplicates:", accounts["account_id"].duplicated().sum())
print("SUBSCRIPTION ID duplicates:", subscriptions["subscription_id"].duplicated().sum())
print("USAGE ID duplicates:", feature_usage["usage_id"].duplicated().sum())
print("TICKET ID duplicates:", support_tickets["ticket_id"].duplicated().sum())
print("CHURN EVENT ID duplicates:", churn_events["churn_event_id"].duplicated().sum())

ACCOUNT ID duplicates: 0
SUBSCRIPTION ID duplicates: 0
USAGE ID duplicates: 21
TICKET ID duplicates: 0
CHURN EVENT ID duplicates: 0


### Primary Key Candidate Review

A duplicate check was performed on the main identifier fields across the five cleaned tables. The results showed that `account_id`, `subscription_id`, `ticket_id`, and `churn_event_id` are fully unique, while `usage_id` in the feature usage table contains 21 duplicate values.

This issue was reviewed in the context of the project design. Since the primary relational keys used in the analytical workflow are `account_id` in the accounts table and `subscription_id` in the subscriptions table, `usage_id` is not treated as a critical structural key in this project. Instead, the feature usage table is analyzed primarily through its relationship to subscriptions via `subscription_id`, which remains the relevant linkage for downstream feature engineering, usage aggregation, and churn analysis.

Therefore, the duplicated `usage_id` values were documented as a non-critical identifier inconsistency rather than a blocking data quality issue. The table remains analytically usable because the forthcoming analysis does not depend on `usage_id` as a unique primary key.
``

## 10.5 Referential Integrity Check

Because the dataset is relational, the next quality step is to validate whether child tables correctly map to their parent tables. This ensures that all subscriptions belong to valid accounts, all feature usage records belong to valid subscriptions, and all support and churn records are attached to valid accounts.

In [77]:
orphan_subscriptions = subscriptions.merge(
    accounts[["account_id"]],
    on="account_id",
    how="left",
    indicator=True
)

print("Orphan subscription rows:", (orphan_subscriptions["_merge"] == "left_only").sum())

Orphan subscription rows: 0


In [78]:
orphan_usage = feature_usage.merge(
    subscriptions[["subscription_id"]],
    on="subscription_id",
    how="left",
    indicator=True
)

print("Orphan feature usage rows:", (orphan_usage["_merge"] == "left_only").sum())

Orphan feature usage rows: 0


In [79]:
orphan_support = support_tickets.merge(
    accounts[["account_id"]],
    on="account_id",
    how="left",
    indicator=True
)

print("Orphan support ticket rows:", (orphan_support["_merge"] == "left_only").sum())

Orphan support ticket rows: 0


In [80]:
orphan_churn = churn_events.merge(
    accounts[["account_id"]],
    on="account_id",
    how="left",
    indicator=True
)

print("Orphan churn event rows:", (orphan_churn["_merge"] == "left_only").sum())

Orphan churn event rows: 0


## 10.6 Temporal Field Re-Parsing and Validation

The initial type-conversion attempt revealed that several core temporal fields were not parsed correctly, which produced artificially high missing-value counts after conversion. To resolve this issue, the cleaned SQL exports are reloaded as strings and re-parsed using more robust helper functions designed to support multiple date and datetime patterns.

This step is critical because temporal fields are central to lifecycle analysis, churn timing analysis, support timeline analysis, and possible cohort-based retention work later in the project.

In [81]:
for name, df in tables.items():
    print(f"\n{name.upper()} - Data Types")
    print(df.dtypes)


ACCOUNTS - Data Types
account_id           str
account_name         str
industry             str
country              str
signup_date          str
referral_source      str
plan_tier            str
seats              int64
is_trial           int64
churn_flag         int64
dtype: object

SUBSCRIPTIONS - Data Types
subscription_id          str
account_id               str
start_date             int64
end_date             float64
plan_tier                str
seats                  int64
mrr_amount             int64
arr_amount             int64
is_trial               int64
upgrade_flag           int64
downgrade_flag         int64
churn_flag             int64
billing_frequency        str
auto_renew_flag        int64
dtype: object

FEATURE_USAGE - Data Types
usage_id                 str
subscription_id          str
usage_date             int64
feature_name             str
usage_count            int64
usage_duration_secs    int64
error_count            int64
is_beta_feature        int64
dtype

## Data Type Issues Identified

### Data Type Issues in the Exported Relational Tables

After reviewing the exported relational tables, several data type inconsistencies were identified that could affect downstream analysis, feature engineering, and modeling.

---

### 1. Date columns are not stored as proper datetime types

Several fields that represent dates or timestamps were imported as `string`, `int64`, or `float64` instead of `datetime`.

**Examples:**
- `accounts.signup_date` is stored as `str`
- `subscriptions.start_date` is stored as `int64`
- `subscriptions.end_date` is stored as `float64`
- `feature_usage.usage_date` is stored as `int64`
- `support_tickets.submitted_at` is stored as `int64`
- `support_tickets.closed_at` is stored as `float64`
- `churn_events.churn_date` is stored as `int64`

This is problematic because date fields should be stored as datetime objects to support:

- time-based filtering
- retention and churn analysis
- cohort analysis
- duration calculations
- chronological sorting

In particular, date columns stored as `float64` strongly suggest the presence of missing values, because pandas automatically converts integer columns with nulls to float.

---

### 2. Boolean / binary flags are stored as integers

Several binary columns are currently stored as `int64`, such as:

- `is_trial`
- `churn_flag`
- `upgrade_flag`
- `downgrade_flag`
- `auto_renew_flag`
- `escalation_flag`
- `is_beta_feature`
- `preceding_upgrade_flag`
- `preceding_downgrade_flag`
- `is_reactivation`

Although storing binary flags as `0` and `1` is analytically acceptable, converting them to `boolean` type improves semantic clarity and makes the dataset easier to validate and interpret.

---

### 3. Numeric columns appear structurally correct, but should still be validated

Some numeric fields are already stored in expected numeric types:

- `seats`
- `mrr_amount`
- `arr_amount`
- `usage_count`
- `usage_duration_secs`
- `error_count`
- `resolution_time_hours`
- `first_response_time_minutes`
- `satisfaction_score`
- `refund_amount_usd`

These columns do not show immediate type issues, but they should still be checked for:

- invalid negative values
- impossible business values
- null handling consistency

---

### 4. Text columns are generally correct

String-based descriptive columns such as:

- `account_id`
- `account_name`
- `industry`
- `country`
- `feature_name`
- `priority`
- `reason_code`
- `feedback_text`

are already stored as text, which is appropriate.

However, they may still require cleaning for:

- leading/trailing spaces
- inconsistent capitalization
- empty strings masquerading as valid values

---

## Summary of Main Data Type Corrections Needed

The main corrections required before analysis are:

1. Convert all date-related columns to proper `datetime`
2. Convert binary flag columns from `int64` to `boolean`
3. Preserve numeric measures as numeric types
4. Standardize string columns and replace empty strings with nulls where necessary

These corrections are necessary to ensure the dataset is analytically reliable and ready for exploratory data analysis, churn modeling, and retention segmentation.

In [82]:
# =========================================================
# 10.6 Temporal Field Re-Parsing and Validation
# Final Corrected Full Version
# =========================================================

import pandas as pd
import numpy as np
import warnings

# ---------------------------------------------------------
# 1) Reload the typed SQL exports as strings
# ---------------------------------------------------------
accounts_raw = pd.read_csv("typed_accounts.csv", dtype="string", keep_default_na=False)
subscriptions_raw = pd.read_csv("typed_subscription.csv", dtype="string", keep_default_na=False)
feature_usage_raw = pd.read_csv("typed_feature_usage.csv", dtype="string", keep_default_na=False)
support_tickets_raw = pd.read_csv("typed_support_tickets.csv", dtype="string", keep_default_na=False)
churn_events_raw = pd.read_csv("typed_churn_events.csv", dtype="string", keep_default_na=False)

# ---------------------------------------------------------
# 2) Helper functions
# ---------------------------------------------------------
def clean_empty_strings(df):
    """
    Replace empty / whitespace-only / obvious null-like strings with <NA>.
    """
    return df.replace(
        {
            r'^\s*$': pd.NA,
            "NULL": pd.NA,
            "null": pd.NA,
            "NaN": pd.NA,
            "nan": pd.NA,
            "None": pd.NA,
            "": pd.NA
        },
        regex=True
    )

def normalize_string_series(series):
    """
    Standardize string formatting before parsing.
    """
    s = series.astype("string").str.strip()
    s = s.replace({
        "": pd.NA,
        "NULL": pd.NA,
        "null": pd.NA,
        "NaN": pd.NA,
        "nan": pd.NA,
        "None": pd.NA
    })
    # Handle float-like exported values such as 20240131.0
    s = s.str.replace(r"\.0$", "", regex=True)
    return s

def safe_to_datetime(series, fmt=None, unit=None):
    """
    Safe wrapper around pd.to_datetime with warning suppression.
    """
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", UserWarning)
        return pd.to_datetime(series, format=fmt, unit=unit, errors="coerce")

def parse_any_date(series):
    """
    Strong date parser (date-only target).
    Handles:
    - YYYY-MM-DD
    - YYYY-MM-DD HH:MM:SS
    - YYYY-MM-DD HH:MM
    - YYYY/MM/DD
    - M/D/YYYY
    - M/D/YYYY HH:MM[:SS]
    - YYYYMMDD
    - Unix sec / ms
    - Excel serial dates
    """
    s = normalize_string_series(series)
    result = pd.Series(pd.NaT, index=s.index, dtype="datetime64[ns]")

    # ISO date
    mask = s.str.match(r'^\d{4}-\d{2}-\d{2}$', na=False)
    result.loc[mask] = safe_to_datetime(s.loc[mask], fmt="%Y-%m-%d")

    # ISO datetime with seconds
    mask = s.str.match(r'^\d{4}-\d{2}-\d{2}\s+\d{2}:\d{2}:\d{2}$', na=False)
    if mask.any():
        result.loc[mask] = safe_to_datetime(s.loc[mask], fmt="%Y-%m-%d %H:%M:%S").dt.normalize()

    # ISO datetime without seconds
    mask = s.str.match(r'^\d{4}-\d{2}-\d{2}\s+\d{2}:\d{2}$', na=False)
    if mask.any():
        result.loc[mask] = safe_to_datetime(s.loc[mask], fmt="%Y-%m-%d %H:%M").dt.normalize()

    # ISO slash date
    mask = s.str.match(r'^\d{4}/\d{2}/\d{2}$', na=False)
    result.loc[mask] = safe_to_datetime(s.loc[mask], fmt="%Y/%m/%d")

    # US date
    mask = s.str.match(r'^\d{1,2}/\d{1,2}/\d{4}$', na=False)
    result.loc[mask] = safe_to_datetime(s.loc[mask], fmt="%m/%d/%Y")

    # US datetime
    mask = s.str.match(r'^\d{1,2}/\d{1,2}/\d{4}\s+\d{1,2}:\d{2}(:\d{2})?$', na=False)
    if mask.any():
        result.loc[mask] = safe_to_datetime(s.loc[mask]).dt.normalize()

    # YYYYMMDD
    mask = s.str.match(r'^\d{8}$', na=False)
    result.loc[mask] = safe_to_datetime(s.loc[mask], fmt="%Y%m%d")

    # Unix timestamps in seconds
    mask = s.str.match(r'^\d{10}$', na=False)
    if mask.any():
        result.loc[mask] = safe_to_datetime(
            s.loc[mask].astype("int64"),
            unit="s"
        ).dt.normalize()

    # Unix timestamps in milliseconds
    mask = s.str.match(r'^\d{13}$', na=False)
    if mask.any():
        result.loc[mask] = safe_to_datetime(
            s.loc[mask].astype("int64"),
            unit="ms"
        ).dt.normalize()

    # Excel serial date numbers
    num = pd.to_numeric(s, errors="coerce")
    mask_excel = num.between(20000, 60000, inclusive="both") & result.isna()
    if mask_excel.any():
        result.loc[mask_excel] = pd.to_datetime(
            num.loc[mask_excel],
            unit="D",
            origin="1899-12-30",
            errors="coerce"
        )

    # Final fallback
    remaining = result.isna() & s.notna()
    if remaining.any():
        try:
            result.loc[remaining] = pd.to_datetime(
                s.loc[remaining],
                format="mixed",
                errors="coerce"
            ).normalize()
        except Exception:
            result.loc[remaining] = safe_to_datetime(s.loc[remaining]).dt.normalize()

    return result

def parse_any_datetime(series):
    """
    Strong datetime parser.
    Handles:
    - YYYY-MM-DD HH:MM:SS
    - YYYY-MM-DD HH:MM
    - YYYY-MM-DD
    - YYYY/MM/DD
    - M/D/YYYY
    - M/D/YYYY HH:MM[:SS]
    - YYYYMMDD
    - Unix sec / ms
    - Excel serial dates
    """
    s = normalize_string_series(series)
    result = pd.Series(pd.NaT, index=s.index, dtype="datetime64[ns]")

    # ISO datetime with seconds
    mask = s.str.match(r'^\d{4}-\d{2}-\d{2}\s+\d{2}:\d{2}:\d{2}$', na=False)
    result.loc[mask] = safe_to_datetime(s.loc[mask], fmt="%Y-%m-%d %H:%M:%S")

    # ISO datetime without seconds
    mask = s.str.match(r'^\d{4}-\d{2}-\d{2}\s+\d{2}:\d{2}$', na=False)
    result.loc[mask] = safe_to_datetime(s.loc[mask], fmt="%Y-%m-%d %H:%M")

    # ISO date only
    mask = s.str.match(r'^\d{4}-\d{2}-\d{2}$', na=False)
    result.loc[mask] = safe_to_datetime(s.loc[mask], fmt="%Y-%m-%d")

    # ISO slash date
    mask = s.str.match(r'^\d{4}/\d{2}/\d{2}$', na=False)
    result.loc[mask] = safe_to_datetime(s.loc[mask], fmt="%Y/%m/%d")

    # US date
    mask = s.str.match(r'^\d{1,2}/\d{1,2}/\d{4}$', na=False)
    result.loc[mask] = safe_to_datetime(s.loc[mask], fmt="%m/%d/%Y")

    # US datetime
    mask = s.str.match(r'^\d{1,2}/\d{1,2}/\d{4}\s+\d{1,2}:\d{2}(:\d{2})?$', na=False)
    if mask.any():
        result.loc[mask] = safe_to_datetime(s.loc[mask])

    # YYYYMMDD
    mask = s.str.match(r'^\d{8}$', na=False)
    result.loc[mask] = safe_to_datetime(s.loc[mask], fmt="%Y%m%d")

    # Unix timestamps in seconds
    mask = s.str.match(r'^\d{10}$', na=False)
    if mask.any():
        result.loc[mask] = safe_to_datetime(
            s.loc[mask].astype("int64"),
            unit="s"
        )

    # Unix timestamps in milliseconds
    mask = s.str.match(r'^\d{13}$', na=False)
    if mask.any():
        result.loc[mask] = safe_to_datetime(
            s.loc[mask].astype("int64"),
            unit="ms"
        )

    # Excel serial date numbers
    num = pd.to_numeric(s, errors="coerce")
    mask_excel = num.between(20000, 60000, inclusive="both") & result.isna()
    if mask_excel.any():
        result.loc[mask_excel] = pd.to_datetime(
            num.loc[mask_excel],
            unit="D",
            origin="1899-12-30",
            errors="coerce"
        )

    # Final fallback
    remaining = result.isna() & s.notna()
    if remaining.any():
        try:
            result.loc[remaining] = pd.to_datetime(
                s.loc[remaining],
                format="mixed",
                errors="coerce"
            )
        except Exception:
            result.loc[remaining] = safe_to_datetime(s.loc[remaining])

    return result

def to_nullable_bool(series):
    """
    Convert 0/1, True/False, yes/no, y/n to pandas nullable boolean.
    """
    s = normalize_string_series(series).str.lower()
    s = s.replace({
        "1": True,
        "true": True,
        "yes": True,
        "y": True,
        "0": False,
        "false": False,
        "no": False,
        "n": False
    })
    return s.astype("boolean")

# ---------------------------------------------------------
# 3) Clean raw string tables
# ---------------------------------------------------------
accounts_raw = clean_empty_strings(accounts_raw)
subscriptions_raw = clean_empty_strings(subscriptions_raw)
feature_usage_raw = clean_empty_strings(feature_usage_raw)
support_tickets_raw = clean_empty_strings(support_tickets_raw)
churn_events_raw = clean_empty_strings(churn_events_raw)

# Keep working copies
accounts = accounts_raw.copy()
subscriptions = subscriptions_raw.copy()
feature_usage = feature_usage_raw.copy()
support_tickets = support_tickets_raw.copy()
churn_events = churn_events_raw.copy()

# ---------------------------------------------------------
# 4) Apply corrected parsing and type conversion
# ---------------------------------------------------------

# ACCOUNTS
accounts["signup_date"] = parse_any_date(accounts_raw["signup_date"])
accounts["is_trial"] = to_nullable_bool(accounts_raw["is_trial"])
accounts["churn_flag"] = to_nullable_bool(accounts_raw["churn_flag"])
accounts["seats"] = pd.to_numeric(accounts_raw["seats"], errors="coerce").astype("Int64")

# SUBSCRIPTIONS
subscriptions["start_date"] = parse_any_date(subscriptions_raw["start_date"])
subscriptions["end_date"] = parse_any_date(subscriptions_raw["end_date"])

for col in ["is_trial", "upgrade_flag", "downgrade_flag", "churn_flag", "auto_renew_flag"]:
    subscriptions[col] = to_nullable_bool(subscriptions_raw[col])

subscriptions["seats"] = pd.to_numeric(subscriptions_raw["seats"], errors="coerce").astype("Int64")
subscriptions["mrr_amount"] = pd.to_numeric(subscriptions_raw["mrr_amount"], errors="coerce").astype("Int64")
subscriptions["arr_amount"] = pd.to_numeric(subscriptions_raw["arr_amount"], errors="coerce").astype("Int64")

# FEATURE USAGE
feature_usage["usage_date"] = parse_any_date(feature_usage_raw["usage_date"])
feature_usage["is_beta_feature"] = to_nullable_bool(feature_usage_raw["is_beta_feature"])

for col in ["usage_count", "usage_duration_secs", "error_count"]:
    feature_usage[col] = pd.to_numeric(feature_usage_raw[col], errors="coerce").astype("Int64")

# SUPPORT TICKETS
support_tickets["submitted_at"] = parse_any_datetime(support_tickets_raw["submitted_at"])
support_tickets["closed_at"] = parse_any_datetime(support_tickets_raw["closed_at"])
support_tickets["escalation_flag"] = to_nullable_bool(support_tickets_raw["escalation_flag"])
support_tickets["resolution_time_hours"] = pd.to_numeric(
    support_tickets_raw["resolution_time_hours"], errors="coerce"
)
support_tickets["first_response_time_minutes"] = pd.to_numeric(
    support_tickets_raw["first_response_time_minutes"], errors="coerce"
).astype("Int64")
support_tickets["satisfaction_score"] = pd.to_numeric(
    support_tickets_raw["satisfaction_score"], errors="coerce"
).astype("Int64")

# CHURN EVENTS
churn_events["churn_date"] = parse_any_date(churn_events_raw["churn_date"])

for col in ["preceding_upgrade_flag", "preceding_downgrade_flag", "is_reactivation"]:
    churn_events[col] = to_nullable_bool(churn_events_raw[col])

churn_events["refund_amount_usd"] = pd.to_numeric(
    churn_events_raw["refund_amount_usd"], errors="coerce"
)

# ---------------------------------------------------------
# 5) Validation - corrected dtypes
# ---------------------------------------------------------
print("=== Corrected Data Types ===\n")

for name, df in {
    "accounts": accounts,
    "subscriptions": subscriptions,
    "feature_usage": feature_usage,
    "support_tickets": support_tickets,
    "churn_events": churn_events
}.items():
    print(f"\n{name.upper()} - Corrected Data Types")
    print(df.dtypes)

# ---------------------------------------------------------
# 6) Validation - missing values after corrected conversion
# ---------------------------------------------------------
print("\n=== Missing Values After Corrected Type Conversion ===\n")

for name, df in {
    "accounts": accounts,
    "subscriptions": subscriptions,
    "feature_usage": feature_usage,
    "support_tickets": support_tickets,
    "churn_events": churn_events
}.items():
    print(f"\n{name.upper()} - Missing Values After Corrected Type Conversion")
    missing = df.isnull().sum()
    print(missing[missing > 0] if missing.sum() > 0 else "No missing values found.")

# ---------------------------------------------------------
# 7) Temporal consistency checks
# ---------------------------------------------------------
print("\n=== Temporal Consistency Checks ===\n")

print("Invalid subscription dates (start_date > end_date):")
print(
    subscriptions[
        (subscriptions["end_date"].notna()) &
        (subscriptions["start_date"] > subscriptions["end_date"])
    ].shape[0]
)

print("\nInvalid support ticket dates (submitted_at > closed_at):")
print(
    support_tickets[
        (support_tickets["closed_at"].notna()) &
        (support_tickets["submitted_at"] > support_tickets["closed_at"])
    ].shape[0]
)

# ---------------------------------------------------------
# 8) Diagnostics - show TRUE unparsed raw values
# ---------------------------------------------------------
def show_unparsed_values(raw_series, parsed_series, n=20):
    s = normalize_string_series(raw_series)
    bad = s[parsed_series.isna() & s.notna()].drop_duplicates()
    return bad.head(n)

print("\n=== True Unparsed Raw Values ===\n")

print("Unparsed subscriptions.start_date values:")
print(show_unparsed_values(subscriptions_raw["start_date"], subscriptions["start_date"]))

print("\nUnparsed subscriptions.end_date values:")
print(show_unparsed_values(subscriptions_raw["end_date"], subscriptions["end_date"]))

print("\nUnparsed feature_usage.usage_date values:")
print(show_unparsed_values(feature_usage_raw["usage_date"], feature_usage["usage_date"]))

print("\nUnparsed support_tickets.submitted_at values:")
print(show_unparsed_values(support_tickets_raw["submitted_at"], support_tickets["submitted_at"]))

print("\nUnparsed support_tickets.closed_at values:")
print(show_unparsed_values(support_tickets_raw["closed_at"], support_tickets["closed_at"]))

print("\nUnparsed churn_events.churn_date values:")
print(show_unparsed_values(churn_events_raw["churn_date"], churn_events["churn_date"]))

# ---------------------------------------------------------
# 9) Final clean copies for next stage
# ---------------------------------------------------------
accounts_clean = accounts.copy()
subscriptions_clean = subscriptions.copy()
feature_usage_clean = feature_usage.copy()
support_tickets_clean = support_tickets.copy()
churn_events_clean = churn_events.copy()

print("\n=== Clean copies created successfully ===")

=== Corrected Data Types ===


ACCOUNTS - Corrected Data Types
account_id                 string
account_name               string
industry                   string
country                    string
signup_date        datetime64[ns]
referral_source            string
plan_tier                  string
seats                       Int64
is_trial                  boolean
churn_flag                boolean
dtype: object

SUBSCRIPTIONS - Corrected Data Types
subscription_id              string
account_id                   string
start_date           datetime64[ns]
end_date             datetime64[ns]
plan_tier                    string
seats                         Int64
mrr_amount                    Int64
arr_amount                    Int64
is_trial                    boolean
upgrade_flag                boolean
downgrade_flag              boolean
churn_flag                  boolean
billing_frequency            string
auto_renew_flag             boolean
dtype: object

FEATURE_USAGE - Corrected 

## 10.7 Date Parsing and Temporal Validation

Time-related fields are critical in churn analysis because customer behavior must be interpreted across the customer lifecycle. After applying the corrected parsing logic, this step validates that the main temporal fields are stored correctly and that their chronological relationships remain logically consistent.

The checks in this section focus on:
- verifying that the parsed date/time fields contain only business-expected missing values
- confirming that subscription start dates do not occur after subscription end dates
- confirming that support ticket submission timestamps do not occur after closure timestamps

This step ensures that the dataset is temporally reliable before moving into exploratory data analysis, retention analysis, and churn interpretation.


In [83]:
# ---------------------------------------------------------
# 10.7 Date Parsing and Temporal Validation
# ---------------------------------------------------------

print("=== Temporal Missing Value Validation ===\n")

temporal_missing = {
    "accounts.signup_date": accounts["signup_date"].isna().sum(),
    "subscriptions.start_date": subscriptions["start_date"].isna().sum(),
    "subscriptions.end_date": subscriptions["end_date"].isna().sum(),
    "feature_usage.usage_date": feature_usage["usage_date"].isna().sum(),
    "support_tickets.submitted_at": support_tickets["submitted_at"].isna().sum(),
    "support_tickets.closed_at": support_tickets["closed_at"].isna().sum(),
    "churn_events.churn_date": churn_events["churn_date"].isna().sum()
}

for col, value in temporal_missing.items():
    print(f"{col}: {value}")

print("\n=== Temporal Consistency Checks ===\n")

print("Invalid subscription dates (start_date > end_date):")
invalid_subscription_dates = subscriptions[
    (subscriptions["end_date"].notna()) &
    (subscriptions["start_date"] > subscriptions["end_date"])
]
print(invalid_subscription_dates.shape[0])

print("\nInvalid support ticket dates (submitted_at > closed_at):")
invalid_support_dates = support_tickets[
    (support_tickets["closed_at"].notna()) &
    (support_tickets["submitted_at"] > support_tickets["closed_at"])
]
print(invalid_support_dates.shape[0])

print("\n=== Sample Invalid Rows (if any) ===\n")

print("Sample invalid subscription date rows:")
display(
    invalid_subscription_dates[
        ["subscription_id", "account_id", "start_date", "end_date"]
    ].head(10)
)

print("Sample invalid support ticket date rows:")
display(
    invalid_support_dates[
        ["ticket_id", "account_id", "submitted_at", "closed_at"]
    ].head(10)
)

=== Temporal Missing Value Validation ===

accounts.signup_date: 0
subscriptions.start_date: 0
subscriptions.end_date: 4514
feature_usage.usage_date: 0
support_tickets.submitted_at: 0
support_tickets.closed_at: 0
churn_events.churn_date: 0

=== Temporal Consistency Checks ===

Invalid subscription dates (start_date > end_date):
0

Invalid support ticket dates (submitted_at > closed_at):
0

=== Sample Invalid Rows (if any) ===

Sample invalid subscription date rows:


,subscription_id,account_id,start_date,end_date


Sample invalid support ticket date rows:


,ticket_id,account_id,submitted_at,closed_at


### Temporal Validation Summary

The temporal validation confirms that the parsed date/time fields are now usable for downstream analysis. No invalid chronological ordering was detected between subscription start and end dates, and no support ticket records were found where submission occurred after closure. The remaining missing values in temporal columns are limited to business-expected cases, such as open-ended subscriptions with no recorded end date. This indicates that the dataset is now temporally ready for exploratory analysis and subsequent churn-related modeling.

## Downstream Analysis Setup

Before continuing the remaining quality-validity checks, the cleaned DataFrames produced after the corrected temporal parsing stage are synchronized into a dedicated dictionary. This ensures that all subsequent validations are performed on the final corrected tables rather than on older intermediate objects.


In [98]:
# =========================================================
# Sync cleaned DataFrames for all downstream sections (10.8+)
# =========================================================
tables_clean = {
    "accounts": accounts,
    "subscriptions": subscriptions,
    "feature_usage": feature_usage,
    "support_tickets": support_tickets,
    "churn_events": churn_events
}

# Optional aliases for readability
accounts_cur = accounts
subscriptions_cur = subscriptions
feature_usage_cur = feature_usage
support_tickets_cur = support_tickets
churn_events_cur = churn_events

print("Cleaned analytical tables are synchronized successfully.")

Cleaned analytical tables are synchronized successfully.


## 10.8 Binary and Categorical Value Validation

Binary flags and categorical fields are reviewed to ensure they contain valid and interpretable values. This step confirms that the cleaned tables preserve meaningful business categories and that binary variables remain consistent for segmentation, churn comparison, and downstream feature engineering.

In [99]:
binary_columns = {
    "accounts": ["is_trial", "churn_flag"],
    "subscriptions": ["is_trial", "upgrade_flag", "downgrade_flag", "churn_flag", "auto_renew_flag"],
    "feature_usage": ["is_beta_feature"],
    "support_tickets": ["escalation_flag"],
    "churn_events": ["preceding_upgrade_flag", "preceding_downgrade_flag", "is_reactivation"]
}

for table_name, cols in binary_columns.items():
    print(f"\n{table_name.upper()}")
    df = tables_clean[table_name]
    for col in cols:
        print(f"{col}: {df[col].dropna().unique()}")


ACCOUNTS
is_trial: [0 1]
churn_flag: [0 1]

SUBSCRIPTIONS
is_trial: [0 1]
upgrade_flag: [0 1]
downgrade_flag: [0 1]
churn_flag: [1 0]
auto_renew_flag: [1 0]

FEATURE_USAGE
is_beta_feature: [0 1]

SUPPORT_TICKETS
escalation_flag: [0 1]

CHURN_EVENTS
preceding_upgrade_flag: [0 1]
preceding_downgrade_flag: [0 1]
is_reactivation: [0 1]


### Binary Validation Summary

The binary validation confirms that the cleaned tables preserve the expected two-state logic across the dataset. All binary fields remain analytically interpretable and suitable for downstream segmentation, group comparison, and churn-related analysis.

In [100]:
categorical_columns = {
    "accounts": ["industry", "country", "referral_source", "plan_tier"],
    "subscriptions": ["plan_tier", "billing_frequency"],
    "support_tickets": ["priority"],
    "churn_events": ["reason_code"]
}

for table_name, cols in categorical_columns.items():
    print(f"\n{table_name.upper()}")
    df = tables_clean[table_name]
    for col in cols:
        print(f"\n{col}:")
        display(df[col].dropna().value_counts().head(10))


ACCOUNTS

industry:


industry
DevTools         113
FinTech          112
Cybersecurity    100
HealthTech        96
EdTech            79
Name: count, dtype: int64


country:


country
US    291
UK     58
IN     49
AU     32
DE     25
CA     23
FR     22
Name: count, dtype: int64


referral_source:


...

billing_frequency
monthly    2539
annual     2461
Name: count, dtype: int64


SUPPORT_TICKETS

priority:


priority
urgent    514
high      510
medium    491
low       485
Name: count, dtype: int64


CHURN_EVENTS

reason_code:


reason_code
features      114
support       104
budget        104
unknown        95
competitor     92
pricing        91
Name: count, dtype: int64

### Categorical Validation Summary

The categorical distributions remain stable after the temporal parsing correction, which confirms that the parsing fix did not distort the business structure of the dataset. Core dimensions such as industry, country, plan tier, billing frequency, support priority, and churn reason continue to show clear and interpretable distributions suitable for downstream exploratory analysis.

## 10.9 Numeric Range Validation

Numeric fields are is to identify impossible values, unexpected outliers, or conversion-related inconsistencies before moving into exploratory analysis and churn-focused interpretation.

In [101]:
numeric_columns = {
    "accounts": ["seats"],
    "subscriptions": ["seats", "mrr_amount", "arr_amount"],
    "feature_usage": ["usage_count", "usage_duration_secs", "error_count"],
    "support_tickets": ["resolution_time_hours", "first_response_time_minutes", "satisfaction_score"],
    "churn_events": ["refund_amount_usd"]
}

for table_name, cols in numeric_columns.items():
    print(f"\n{table_name.upper()}")
    df = tables_clean[table_name]
    display(df[cols].describe(include="all"))


ACCOUNTS


,seats
count,500.000000
mean,20.560000
std,21.044718
min,1.000000
25%,5.000000
50%,15.000000
75%,28.000000
max,163.000000



SUBSCRIPTIONS


,seats,mrr_amount,arr_amount
count,5000.000000,5000.000000,5000.000000
mean,29.852000,2267.749400,27212.992800
std,23.089771,3421.375348,41056.504178
min,1.000000,0.000000,0.000000
25%,14.000000,285.000000,3420.000000
50%,24.000000,931.000000,11172.000000
75%,40.000000,2786.000000,33432.000000
max,189.000000,33830.000000,405960.000000



FEATURE_USAGE


,usage_count,usage_duration_secs,error_count
count,25000.000000,25000.000000,25000.000000
mean,10.021000,3042.202880,0.564280
std,3.143729,2056.544615,1.012595
min,0.000000,0.000000,0.000000
25%,8.000000,1350.000000,0.000000
50%,10.000000,2760.000000,0.000000
75%,12.000000,4400.000000,1.000000
max,26.000000,12696.000000,8.000000



SUPPORT_TICKETS


,resolution_time_hours,first_response_time_minutes,satisfaction_score
count,2000.000000,2000.000000,2000.000000
mean,35.861000,88.480000,2.339000
std,21.138427,51.531877,2.056257
min,1.000000,1.000000,0.000000
25%,17.000000,43.000000,0.000000
50%,35.000000,88.000000,3.000000
75%,54.000000,131.000000,4.000000
max,72.000000,180.000000,5.000000



CHURN_EVENTS


,refund_amount_usd
count,600.000000
mean,14.420417
std,39.224591
min,0.000000
25%,0.000000
50%,0.000000
75%,0.000000
max,392.920000


### Numeric Range Validation Summary

The numeric range review does not reveal any major data quality issues. Overall, the values appear to fall within realistic and interpretable business ranges.

Most numeric fields show reasonable minimums, maximums, and distributions:

- `seats` in both **accounts** and **subscriptions** appears plausible.
- `mrr_amount` and `arr_amount` are positive and show expected right-skewed business distributions.
- `usage_count`, `usage_duration_secs`, and `error_count` fall within believable usage ranges.
- `resolution_time_hours` and `first_response_time_minutes` are also within realistic support operation limits.
- `refund_amount_usd` is highly concentrated at zero, which is expected if many churn events do not result in refunds.

However, a few values should be reviewed against business rules:

- `satisfaction_score` has a minimum value of `0`.  
  This is acceptable only if the scoring scale is defined as **0–5**. If the intended scale is **1–5**, then this requires correction.

- `mrr_amount` and `arr_amount` have minimum values of `0`.  
  These may be valid for trial, inactive, or zero-revenue subscriptions, but they should be confirmed against the pricing model.

- `usage_count` and `usage_duration_secs` also include `0` values.  
  These may be legitimate, but they should be checked to ensure they do not result from parsing issues or invalid event records.

In summary, the numeric fields are generally consistent and analytically usable. No strong evidence of impossible values is visible, although a few boundary cases should be validated using business context.


# 10.10 Targeted Business-Context Validation

This notebook section validates a small set of boundary-case numeric values that may still require business confirmation even though the overall numeric distributions appear reasonable.

## Validation focus

The following values will be reviewed:

1. `support_tickets.satisfaction_score == 0`
2. `subscriptions.mrr_amount == 0` or `subscriptions.arr_amount == 0`
3. `feature_usage.usage_count == 0` or `feature_usage.usage_duration_secs == 0`
4. `subscriptions.arr_amount` consistency with `mrr_amount * 12`

## Objective

The goal is to determine whether these values are:
- valid business cases,
- acceptable edge cases,
- or potential data quality issues requiring correction.

In [102]:
# =========================================================
# Targeted checks - use cleaned DataFrames only
# =========================================================
accounts = accounts_cur
subscriptions = subscriptions_cur
feature_usage = feature_usage_cur
support_tickets = support_tickets_cur
churn_events = churn_events_cur

print("All required cleaned DataFrames are available.")

All required cleaned DataFrames are available.


### Step 1 — Quick Validation Snapshot

This step provides a fast overview of the main boundary-case values that will be examined in more detail.

In [103]:
print("=== Quick Validation Summary ===")

print("\n[Support Tickets]")
print("satisfaction_score == 0:", (support_tickets["satisfaction_score"] == 0).sum())

print("\n[Subscriptions]")
print("mrr_amount == 0:", (subscriptions["mrr_amount"] == 0).sum())
print("arr_amount == 0:", (subscriptions["arr_amount"] == 0).sum())
print(
    "zero revenue rows:",
    ((subscriptions["mrr_amount"] == 0) | (subscriptions["arr_amount"] == 0)).sum()
)

print("\n[Feature Usage]")
print("usage_count == 0:", (feature_usage["usage_count"] == 0).sum())
print("usage_duration_secs == 0:", (feature_usage["usage_duration_secs"] == 0).sum())
print(
    "both usage_count == 0 and usage_duration_secs == 0:",
    ((feature_usage["usage_count"] == 0) & (feature_usage["usage_duration_secs"] == 0)).sum()
)

=== Quick Validation Summary ===

[Support Tickets]
satisfaction_score == 0: 825

[Subscriptions]
mrr_amount == 0: 778
arr_amount == 0: 778
zero revenue rows: 778

[Feature Usage]
usage_count == 0: 2
usage_duration_secs == 0: 2
both usage_count == 0 and usage_duration_secs == 0: 2


### Step 2 — Validate `satisfaction_score`

A minimum satisfaction score of 0 may be valid if the scoring scale is defined as 0–5. However, if the intended scale is 1–5, then a score of 0 should be treated as invalid or as a “not rated” category. This distinction is important because it directly affects support-quality metrics and any later churn interpretation involving customer satisfaction.

In [104]:
# 2.1 Distribution of satisfaction scores
score_dist = support_tickets["satisfaction_score"].value_counts(dropna=False).sort_index()
print("Distribution of satisfaction_score:")
display(score_dist)

# 2.2 Show sample rows where satisfaction_score == 0
score_zero_sample = support_tickets.loc[
    support_tickets["satisfaction_score"] == 0,
    ["ticket_id", "submitted_at", "closed_at", "priority", "satisfaction_score"]
].head(10)

print("Sample rows where satisfaction_score == 0:")
display(score_zero_sample)

# 2.3 Is score 0 associated with closed tickets?
closure_by_score = (
    support_tickets.assign(is_closed=support_tickets["closed_at"].notna())
    .groupby("satisfaction_score")["is_closed"]
    .mean()
    .rename("share_closed")
)

print("Share of closed tickets by satisfaction score:")
display(closure_by_score)

# 2.4 Strict check if expected valid scale is 1–5 only
invalid_scores_if_1_to_5 = support_tickets.loc[
    ~support_tickets["satisfaction_score"].between(1, 5),
    ["ticket_id", "satisfaction_score"]
]

print("Count of invalid satisfaction scores (if expected scale = 1–5):", len(invalid_scores_if_1_to_5))
display(invalid_scores_if_1_to_5.head(10))

Distribution of satisfaction_score:


satisfaction_score
0    825
3    396
4    405
5    374
Name: count, dtype: int64

Sample rows where satisfaction_score == 0:


,ticket_id,submitted_at,closed_at,priority,satisfaction_score
0,T-0024de,45134,45135.12500,high,0
1,T-4d04b9,45481,45482.12500,urgent,0
4,T-c59f77,45626,45627.08333,medium,0
5,T-90f06d,45134,45134.37500,medium,0
8,T-7119c9,45165,45165.66667,urgent,0
10,T-98a84a,45280,45281.75000,low,0
12,T-6bd715,45426,45428.08333,medium,0
15,T-f8e2c3,45352,45354.87500,high,0
16,T-4e7816,45568,45569.50000,low,0
18,T-ca96ba,45410,45410.12500,high,0


Share of closed tickets by satisfaction score:


satisfaction_score
0    1.0
3    1.0
4    1.0
5    1.0
Name: share_closed, dtype: float64

Count of invalid satisfaction scores (if expected scale = 1–5): 825


,ticket_id,satisfaction_score
0,T-0024de,0
1,T-4d04b9,0
4,T-c59f77,0
5,T-90f06d,0
8,T-7119c9,0
10,T-98a84a,0
12,T-6bd715,0
15,T-f8e2c3,0
16,T-4e7816,0
18,T-ca96ba,0


#### Interpretation of `satisfaction_score = 0`

The validation results show that `satisfaction_score = 0` occurs frequently in the dataset and should not be interpreted through unresolved-ticket behavior, because the closure check confirms that these cases are not predominantly open or incomplete. Instead, the pattern should be treated as a business-rule question: either `0` is a valid point on the scoring scale, or it is being used as a placeholder for not-rated feedback.

For the purposes of this project, `satisfaction_score = 0` will be treated analytically as **not rated** rather than as a true dissatisfaction score, unless business documentation explicitly confirms that the intended support rating scale is 0–5. This prevents customer satisfaction metrics from being artificially biased downward by values that may reflect missing feedback rather than genuine customer sentiment.

#### Analytical treatment adopted in this study

For the purposes of analysis, `satisfaction_score = 0` will be treated as **not rated** rather than as a true dissatisfaction score. This decision was made to avoid introducing bias into customer satisfaction metrics in the absence of confirmed business documentation stating that `0` is a valid value on the scoring scale.

This treatment is methodologically appropriate because repeated `0` values may reflect missing feedback, unanswered surveys, or system-default placeholders rather than genuine customer evaluations. Including these records as actual satisfaction scores would artificially depress average satisfaction levels and could lead to misleading comparisons across accounts, ticket segments, or churn groups.

To preserve data integrity, the original raw values will remain unchanged in the source dataset, while the analytical version of the variable will interpret `0` as **not rated / missing**. This ensures that satisfaction-based analysis reflects only meaningful customer responses while maintaining traceability to the original data.

In [105]:
support_tickets["satisfaction_score_analytical"] = support_tickets["satisfaction_score"].mask(
    support_tickets["satisfaction_score"] == 0,
    pd.NA
).astype("Int64")

print("Original satisfaction_score distribution:")
display(support_tickets["satisfaction_score"].value_counts(dropna=False).sort_index())

print("\nAnalytical satisfaction_score distribution (0 treated as missing):")
display(support_tickets["satisfaction_score_analytical"].value_counts(dropna=False).sort_index())

Original satisfaction_score distribution:


satisfaction_score
0    825
3    396
4    405
5    374
Name: count, dtype: int64


Analytical satisfaction_score distribution (0 treated as missing):


satisfaction_score_analytical
3       396
4       405
5       374
<NA>    825
Name: count, dtype: Int64

## Step 3 — Validate Zero-Revenue Subscriptions

### Why this matters

A value of `0` for `mrr_amount` or `arr_amount` may be valid for:
- trial subscriptions,
- free plans,
- temporarily inactive subscriptions,
- or certain churned accounts.

However, if zero revenue appears frequently in active, paid subscriptions, it may indicate a parsing issue or a pricing inconsistency.

In [106]:
zero_revenue_mask = (subscriptions["mrr_amount"] == 0) | (subscriptions["arr_amount"] == 0)

zero_revenue_rows = subscriptions.loc[
    zero_revenue_mask,
    ["subscription_id", "account_id", "plan_tier", "is_trial", "churn_flag",
     "billing_frequency", "mrr_amount", "arr_amount"]
].copy()

print("Sample zero-revenue subscription rows:")
display(zero_revenue_rows.head(10))

print("\nZero revenue by trial status:")
display(
    subscriptions.assign(zero_revenue=zero_revenue_mask)
    .groupby("is_trial")["zero_revenue"]
    .agg(["mean", "sum", "count"])
)

print("\nZero revenue by churn status:")
display(
    subscriptions.assign(zero_revenue=zero_revenue_mask)
    .groupby("churn_flag")["zero_revenue"]
    .agg(["mean", "sum", "count"])
)

print("\nZero revenue by plan tier:")
display(
    subscriptions.assign(zero_revenue=zero_revenue_mask)
    .groupby("plan_tier")["zero_revenue"]
    .agg(["mean", "sum", "count"])
    .sort_values("sum", ascending=False)
)

Sample zero-revenue subscription rows:


,subscription_id,account_id,plan_tier,is_trial,churn_flag,billing_frequency,mrr_amount,arr_amount
2,S-51c0d1,A-659280,Enterprise,1,0,annual,0,0
11,S-79a9e1,A-3a5ad8,Enterprise,1,0,annual,0,0
14,S-428e9a,A-8ae3fc,Enterprise,1,0,monthly,0,0
16,S-13939b,A-86902e,Pro,1,1,monthly,0,0
20,S-c27134,A-dec005,Pro,1,0,monthly,0,0
32,S-2877e6,A-95b24a,Enterprise,1,0,monthly,0,0
34,S-d7d398,A-fb186e,Enterprise,1,0,annual,0,0
35,S-ddad91,A-40906c,Enterprise,1,0,annual,0,0
37,S-bb1cf1,A-3a5ad8,Enterprise,1,0,monthly,0,0
49,S-ad39ef,A-e40f45,Pro,1,0,annual,0,0



Zero revenue by trial status:


,mean,sum,count
is_trial,,,
0,0.0,0,4222
1,1.0,778,778



Zero revenue by churn status:


,mean,sum,count
churn_flag,,,
0,0.155073,700,4514
1,0.160494,78,486



Zero revenue by plan tier:


,mean,sum,count
plan_tier,,,
Enterprise,0.158445,273,1723
Pro,0.154627,259,1675
Basic,0.153558,246,1602


#### Interpretation of Zero-Revenue Subscriptions

The zero-revenue pattern remains analytically valid and is fully explained by trial status rather than by a data quality issue. Therefore, zero-revenue subscriptions are retained in the dataset as valid records, but they should be analyzed separately from paid subscriptions whenever revenue-related conclusions are drawn.

### Step 4 — Validate `ARR ≈ MRR × 12`

If `arr_amount` represents annual recurring revenue and `mrr_amount` represents monthly recurring revenue, the expected business relationship is `ARR = MRR × 12`. This step is used to confirm whether the two revenue fields are internally consistent.

In [107]:
subscriptions_check = subscriptions.copy()
subscriptions_check["expected_arr"] = subscriptions_check["mrr_amount"] * 12
subscriptions_check["arr_match"] = subscriptions_check["arr_amount"] == subscriptions_check["expected_arr"]

print("ARR match distribution:")
display(subscriptions_check["arr_match"].value_counts(dropna=False))

print("ARR match rate:", subscriptions_check["arr_match"].mean())

print("Sample ARR mismatches:")
display(
    subscriptions_check.loc[
        ~subscriptions_check["arr_match"],
        ["subscription_id", "mrr_amount", "arr_amount", "expected_arr", "billing_frequency"]
    ].head(10)
)

ARR match distribution:


arr_match
True    5000
Name: count, dtype: int64

ARR match rate: 1.0
Sample ARR mismatches:


,subscription_id,mrr_amount,arr_amount,expected_arr,billing_frequency


### Interpretation of Revenue Consistency 
#### ARR and MRR consistency

The validation of annual and monthly recurring revenue shows perfect internal consistency. All 5,000 subscription records satisfy the expected relationship:

```python
arr_amount = mrr_amount * 12

The ARR match rate of `1.0` indicates that:

- every subscription record has a correctly aligned annual revenue value,
- there are no detected mismatches between `mrr_amount` and `arr_amount`,
- and no evidence of calculation, parsing, or conversion errors is present in these two fields.

The absence of any mismatch cases confirms that the revenue measures are structurally reliable and analytically trustworthy.

## Overall conclusion

Taken together, these results show that the revenue fields are of high quality and do not currently require correction.

More specifically:

- the observed zero-revenue subscriptions are fully explained by trial status,
- they are not primarily driven by plan tier,
- they are not meaningfully associated with churn status,
- and the relationship between `mrr_amount` and `arr_amount` is perfectly consistent across all records.

Therefore, zero-revenue subscriptions should not be treated as erroneous observations. Instead, they should be interpreted as valid trial-related records that reflect expected business behavior.

## Analytical implication

Although zero-revenue subscriptions are valid, they are structurally different from paid subscriptions and should therefore be handled carefully in downstream analysis.

For analytical purposes:

- Revenue-focused analysis should either exclude trial subscriptions or treat them as a separate segment.
- Churn and retention analysis should account for the fact that trial records have zero revenue by design.
- Plan-tier comparisons should be interpreted carefully, since zero revenue within a plan tier may simply reflect trial assignments rather than true paid-plan monetization behavior.

## Recommended treatment in this study

Based on the validation results, zero-revenue subscriptions will be retained in the dataset as valid records. However, because these cases are entirely attributable to trial subscriptions, paid and trial subscriptions should be analyzed separately whenever revenue interpretation is involved.

This treatment preserves data integrity while ensuring that downstream business conclusions are not distorted by mixing trial and paid revenue behavior in the same analytical group.

## Step 5 — Validate Zero-Usage Records

### Why this matters

A usage record with `usage_count = 0` or `usage_duration_secs = 0` may be valid if it represents:
- a failed attempt,
- a tracked open event without meaningful interaction,
- or another event logging rule.

However, records where:
- `usage_count = 0`
- `usage_duration_secs = 0`
- and `error_count = 0`

may require closer inspection because they do not clearly represent useful activity.

In [108]:
zero_usage_mask = (feature_usage["usage_count"] == 0) | (feature_usage["usage_duration_secs"] == 0)

zero_usage_rows = feature_usage.loc[
    zero_usage_mask,
    ["usage_id", "subscription_id", "usage_date", "feature_name",
     "usage_count", "usage_duration_secs", "error_count", "is_beta_feature"]
].copy()

print("Sample zero-usage rows:")
display(zero_usage_rows.head(10))

print("usage_count == 0:", (feature_usage["usage_count"] == 0).sum())
print("usage_duration_secs == 0:", (feature_usage["usage_duration_secs"] == 0).sum())
print("both usage_count == 0 and usage_duration_secs == 0:",
      ((feature_usage["usage_count"] == 0) & (feature_usage["usage_duration_secs"] == 0)).sum())
print("zero usage with error_count > 0:",
      ((feature_usage["usage_count"] == 0) & (feature_usage["error_count"] > 0)).sum())
print("zero count + zero duration + zero error:",
      ((feature_usage["usage_count"] == 0) &
       (feature_usage["usage_duration_secs"] == 0) &
       (feature_usage["error_count"] == 0)).sum())

print("\nZero usage by feature:")
display(
    feature_usage.assign(
        zero_usage=(feature_usage["usage_count"] == 0) | (feature_usage["usage_duration_secs"] == 0)
    )
    .groupby("feature_name")["zero_usage"]
    .agg(["mean", "sum", "count"])
    .sort_values("sum", ascending=False)
    .head(10)
)

Sample zero-usage rows:


,usage_id,subscription_id,usage_date,feature_name,usage_count,usage_duration_secs,error_count,is_beta_feature
2875,U-a72131,S-9925b9,45139,feature_25,0,0,0,0
3327,U-370ae6,S-c98905,45000,feature_34,0,0,2,1


usage_count == 0: 2
usage_duration_secs == 0: 2
both usage_count == 0 and usage_duration_secs == 0: 2
zero usage with error_count > 0: 1
zero count + zero duration + zero error: 1

Zero usage by feature:


,mean,sum,count
feature_name,,,
feature_25,0.001626,1,615
feature_34,0.001538,1,650
feature_1,0.000000,0,629
feature_29,0.000000,0,640
feature_30,0.000000,0,629
feature_31,0.000000,0,644
feature_32,0.000000,0,659
feature_33,0.000000,0,635
feature_35,0.000000,0,587


### Interpretation of Zero-Usage Records

The zero-usage validation results suggest that these cases are extremely rare and do not represent a widespread structural issue in the `feature_usage` table.

The feature-level summary shows that zero-usage records appear only in a very limited number of features:

- **feature_25:** 1 zero-usage record out of 615 observations (0.16%)
- **feature_34:** 1 zero-usage record out of 650 observations (0.15%)

All other reviewed features show no zero-usage cases in the displayed results.

This indicates that zero-usage events are **not concentrated within a specific feature** and do not appear frequently enough to suggest a systematic tracking or parsing problem. Instead, the pattern appears isolated and minimal in scale.

From a data quality perspective, these findings imply that zero-usage records are unlikely to materially affect downstream analysis. They may reflect occasional failed interactions, low-information events, or minor logging edge cases rather than a broader issue in the data collection process.

### Overall conclusion

The observed zero-usage records do not currently indicate a significant data quality concern. Their frequency is extremely low, and they are not clustered in a way that would suggest a feature-specific instrumentation issue.

### Recommended treatment in this study

These records can be retained in the dataset without immediate correction. However, if a stricter event-quality filter is needed in later stages of analysis, records with:

- `usage_count = 0`
- `usage_duration_secs = 0`
- and `error_count = 0`

may be flagged separately as low-information usage events for optional review or exclusion.

## 10.11 Final Cleaning Decisions

Based on the data quality assessment, the RavenStack dataset is considered analytically usable for downstream exploratory analysis and feature engineering. No critical structural blockers remain at this stage. The main identifier fields are valid at the project-critical level, referential integrity is preserved across the relational model, and the remaining edge cases have been documented and treated according to business context rather than removed indiscriminately.

The main analytical treatment decisions adopted in this study are:

- duplicated `usage_id` values are documented as a non-critical identifier inconsistency because the feature usage table is analyzed through `subscription_id`, not through `usage_id` as a strict primary key
- `satisfaction_score = 0` is treated analytically as “not rated” rather than as a true dissatisfaction score
- zero-revenue subscriptions are retained because they are fully explained by trial status
- rare zero-usage events are retained because they are extremely limited and do not represent a material data quality issue
- temporal fields are retained after corrected parsing and may now be used for lifecycle, retention, and time-based analysis where appropriate

In [109]:
# Final cleaned working copies for the next stage
accounts_clean = accounts.copy()
subscriptions_clean = subscriptions.copy()
feature_usage_clean = feature_usage.copy()
support_tickets_clean = support_tickets.copy()
churn_events_clean = churn_events.copy()

print("Final cleaned working copies created successfully.")

Final cleaned working copies created successfully.


## Section 10 Final Summary

The final data quality assessment confirms that the RavenStack dataset is structurally sound, relationally consistent, and analytically ready for downstream work. The temporal parsing issue identified in the earlier version of the notebook has been successfully resolved, and the core date/time fields are now correctly parsed and validated. The only remaining missing values are business-expected cases, specifically open-ended subscription end dates and optional churn feedback text.

Referential integrity is preserved across the relational model, the main analytical keys remain reliable, the duplicate issue in `usage_id` is non-blocking in the context of the project design, and the main numeric measures remain internally consistent and business-plausible. Zero-revenue subscriptions are fully explained by trial status, and the rare zero-usage records are low-impact edge cases rather than structural defects. Therefore, the dataset is now ready for exploratory data analysis, feature engineering, and churn-focused business interpretation.

In [111]:
# Quick check
print(type(accounts_clean))
print(type(subscriptions_clean))
print(type(feature_usage_clean))
print(type(support_tickets_clean))
print(type(churn_events_clean))

<class 'pandas.DataFrame'>
<class 'pandas.DataFrame'>
<class 'pandas.DataFrame'>
<class 'pandas.DataFrame'>
<class 'pandas.DataFrame'>


In [112]:
print("accounts.signup_date:", accounts_clean["signup_date"].dtype)
print("subscriptions.start_date:", subscriptions_clean["start_date"].dtype)
print("subscriptions.end_date:", subscriptions_clean["end_date"].dtype)
print("feature_usage.usage_date:", feature_usage_clean["usage_date"].dtype)
print("support_tickets.submitted_at:", support_tickets_clean["submitted_at"].dtype)
print("support_tickets.closed_at:", support_tickets_clean["closed_at"].dtype)
print("churn_events.churn_date:", churn_events_clean["churn_date"].dtype)

accounts.signup_date: str
subscriptions.start_date: int64
subscriptions.end_date: float64
feature_usage.usage_date: int64
support_tickets.submitted_at: int64
support_tickets.closed_at: float64
churn_events.churn_date: int64


### Export Final Cleaned Tables (Overwrite with Readable Date Format)

After completing the final cleaning stage, the cleaned analytical tables are exported again using the same file names. Before export, all temporal fields are explicitly formatted as readable date or datetime strings so that the CSV outputs preserve human-readable values without breaking the downstream EDA workflow.


In [114]:
import pandas as pd
import numpy as np

# =========================================================
# 1) Load the current final files (same file names)
# =========================================================
accounts_clean = pd.read_csv("final_accounts_clean.csv")
subscriptions_clean = pd.read_csv("final_subscriptions_clean.csv")
feature_usage_clean = pd.read_csv("final_feature_usage_clean.csv")
support_tickets_clean = pd.read_csv("final_support_tickets_clean.csv")
churn_events_clean = pd.read_csv("final_churn_events_clean.csv")

# =========================================================
# 2) Helper functions to convert mixed date formats
# =========================================================

def convert_mixed_date(series):
    """
    Converts a column that may contain:
    - readable date strings
    - Excel serial numbers (date only)
    - empty values
    Returns datetime64[ns]
    """
    s = series.astype("string").str.strip()
    s = s.replace({"": pd.NA, "nan": pd.NA, "NaN": pd.NA, "None": pd.NA})

    numeric_values = pd.to_numeric(s, errors="coerce")
    result = pd.Series(pd.NaT, index=series.index, dtype="datetime64[ns]")

    # numeric Excel serial dates
    numeric_mask = numeric_values.notna()
    if numeric_mask.any():
        result.loc[numeric_mask] = pd.to_datetime("1899-12-30") + pd.to_timedelta(
            numeric_values.loc[numeric_mask], unit="D"
        )

    # text dates
    text_mask = ~numeric_mask & s.notna()
    if text_mask.any():
        result.loc[text_mask] = pd.to_datetime(s.loc[text_mask], errors="coerce")

    # normalize to date-only
    return result.dt.normalize()


def convert_mixed_datetime(series):
    """
    Converts a column that may contain:
    - readable datetime strings
    - Excel serial numbers with fractional day (preserve time)
    - empty values
    Returns datetime64[ns]
    """
    s = series.astype("string").str.strip()
    s = s.replace({"": pd.NA, "nan": pd.NA, "NaN": pd.NA, "None": pd.NA})

    numeric_values = pd.to_numeric(s, errors="coerce")
    result = pd.Series(pd.NaT, index=series.index, dtype="datetime64[ns]")

    # numeric Excel serial date-times
    numeric_mask = numeric_values.notna()
    if numeric_mask.any():
        result.loc[numeric_mask] = pd.to_datetime("1899-12-30") + pd.to_timedelta(
            numeric_values.loc[numeric_mask], unit="D"
        )

    # text datetimes
    text_mask = ~numeric_mask & s.notna()
    if text_mask.any():
        result.loc[text_mask] = pd.to_datetime(s.loc[text_mask], errors="coerce")

    return result


def format_date_col(series, fmt):
    """
    Format datetime column as readable string while preserving blanks
    """
    return series.dt.strftime(fmt).where(series.notna(), "")

# =========================================================
# 3) Convert date / datetime columns FIRST
# =========================================================

# Accounts
accounts_clean["signup_date"] = convert_mixed_date(accounts_clean["signup_date"])

# Subscriptions
subscriptions_clean["start_date"] = convert_mixed_date(subscriptions_clean["start_date"])
subscriptions_clean["end_date"] = convert_mixed_date(subscriptions_clean["end_date"])

# Feature usage
feature_usage_clean["usage_date"] = convert_mixed_date(feature_usage_clean["usage_date"])

# Support tickets
support_tickets_clean["submitted_at"] = convert_mixed_datetime(support_tickets_clean["submitted_at"])
support_tickets_clean["closed_at"] = convert_mixed_datetime(support_tickets_clean["closed_at"])

# Churn events
churn_events_clean["churn_date"] = convert_mixed_date(churn_events_clean["churn_date"])

# =========================================================
# 4) Check dtypes BEFORE export
# =========================================================
print("=== Fixed Dtypes Before Export ===")
print("accounts.signup_date:", accounts_clean["signup_date"].dtype)
print("subscriptions.start_date:", subscriptions_clean["start_date"].dtype)
print("subscriptions.end_date:", subscriptions_clean["end_date"].dtype)
print("feature_usage.usage_date:", feature_usage_clean["usage_date"].dtype)
print("support_tickets.submitted_at:", support_tickets_clean["submitted_at"].dtype)
print("support_tickets.closed_at:", support_tickets_clean["closed_at"].dtype)
print("churn_events.churn_date:", churn_events_clean["churn_date"].dtype)

# =========================================================
# 5) Create export copies
# =========================================================
accounts_export = accounts_clean.copy()
subscriptions_export = subscriptions_clean.copy()
feature_usage_export = feature_usage_clean.copy()
support_tickets_export = support_tickets_clean.copy()
churn_events_export = churn_events_clean.copy()

# =========================================================
# 6) Format date columns as readable strings
# =========================================================

# Accounts
accounts_export["signup_date"] = format_date_col(accounts_export["signup_date"], "%Y-%m-%d")

# Subscriptions
subscriptions_export["start_date"] = format_date_col(subscriptions_export["start_date"], "%Y-%m-%d")
subscriptions_export["end_date"] = format_date_col(subscriptions_export["end_date"], "%Y-%m-%d")

# Feature usage
feature_usage_export["usage_date"] = format_date_col(feature_usage_export["usage_date"], "%Y-%m-%d")

# Support tickets
support_tickets_export["submitted_at"] = format_date_col(
    support_tickets_export["submitted_at"], "%Y-%m-%d %H:%M:%S"
)
support_tickets_export["closed_at"] = format_date_col(
    support_tickets_export["closed_at"], "%Y-%m-%d %H:%M:%S"
)

# Churn events
churn_events_export["churn_date"] = format_date_col(churn_events_export["churn_date"], "%Y-%m-%d")

# =========================================================
# 7) OVERWRITE the same file names
# =========================================================
accounts_export.to_csv("final_accounts_clean.csv", index=False)
subscriptions_export.to_csv("final_subscriptions_clean.csv", index=False)
feature_usage_export.to_csv("final_feature_usage_clean.csv", index=False)
support_tickets_export.to_csv("final_support_tickets_clean.csv", index=False)
churn_events_export.to_csv("final_churn_events_clean.csv", index=False)

print("\nFinal cleaned CSV files were overwritten successfully.")

=== Fixed Dtypes Before Export ===
accounts.signup_date: datetime64[ns]
subscriptions.start_date: datetime64[ns]
subscriptions.end_date: datetime64[ns]
feature_usage.usage_date: datetime64[ns]
support_tickets.submitted_at: datetime64[ns]
support_tickets.closed_at: datetime64[ns]
churn_events.churn_date: datetime64[ns]

Final cleaned CSV files were overwritten successfully.


In [115]:
print("Accounts Preview:")
display(pd.read_csv("final_accounts_clean.csv").head())

print("Subscriptions Preview:")
display(pd.read_csv("final_subscriptions_clean.csv").head())

print("Feature Usage Preview:")
display(pd.read_csv("final_feature_usage_clean.csv").head())

print("Support Tickets Preview:")
display(pd.read_csv("final_support_tickets_clean.csv").head())

print("Churn Events Preview:")
display(pd.read_csv("final_churn_events_clean.csv").head())

Accounts Preview:


,account_id,account_name,industry,country,signup_date,referral_source,plan_tier,seats,is_trial,churn_flag
0,A-2e4581,Company_0,EdTech,US,2024-10-16,partner,Basic,9,0,0
1,A-43a9e3,Company_1,FinTech,IN,2023-08-17,other,Basic,18,0,1
2,A-0a282f,Company_2,DevTools,US,2024-08-27,organic,Basic,1,0,0
3,A-1f0ac7,Company_3,HealthTech,UK,2023-08-27,other,Basic,24,1,0
4,A-ce550d,Company_4,HealthTech,US,2024-10-27,event,Enterprise,35,0,1


Subscriptions Preview:


,subscription_id,account_id,start_date,end_date,plan_tier,seats,mrr_amount,arr_amount,is_trial,upgrade_flag,downgrade_flag,churn_flag,billing_frequency,auto_renew_flag
0,S-8cec59,A-3c1a3f,2023-12-23,2024-04-12,Enterprise,14,2786,33432,0,0,0,1,monthly,1
1,S-0f6f44,A-9b9fe9,2024-06-11,NaN,Pro,17,833,9996,0,0,0,0,monthly,1
2,S-51c0d1,A-659280,2024-11-25,NaN,Enterprise,62,0,0,1,1,0,0,annual,0
3,S-f81687,A-e7a1e2,2024-11-23,2024-12-13,Enterprise,5,995,11940,0,0,0,1,monthly,1
4,S-cff5a2,A-ba6516,2024-01-10,NaN,Enterprise,27,5373,64476,0,0,0,0,monthly,1


Feature Usage Preview:


,usage_id,subscription_id,usage_date,feature_name,usage_count,usage_duration_secs,error_count,is_beta_feature
0,U-1c6c24,S-0fcf7d,2023-07-27,feature_20,9,5004,0,0
1,U-f07cb8,S-c25263,2023-08-07,feature_5,9,369,0,0
2,U-096807,S-f29e7f,2023-12-07,feature_3,9,1458,0,0
3,U-6b1580,S-be655e,2024-07-28,feature_40,5,2085,0,0
4,U-720a29,S-f9b1d0,2024-12-02,feature_12,12,900,0,0


Support Tickets Preview:


,ticket_id,account_id,submitted_at,closed_at,resolution_time_hours,priority,first_response_time_minutes,satisfaction_score,escalation_flag,satisfaction_score_analytical
0,T-0024de,A-712f1c,2023-07-27 00:00:00,2023-07-28 03:00:00,27,high,74,0,0,NaN
1,T-4d04b9,A-e43bf7,2024-07-08 00:00:00,2024-07-09 03:00:00,27,urgent,144,0,0,NaN
2,T-d5e12f,A-0f3e88,2024-10-17 00:00:00,2024-10-17 19:00:00,19,urgent,93,4,0,4.0
3,T-dfce9a,A-4c56c9,2024-09-08 00:00:00,2024-09-09 22:59:59,47,medium,126,5,0,5.0
4,T-c59f77,A-6f8ad2,2024-11-30 00:00:00,2024-12-01 01:59:59,26,medium,8,0,0,NaN


Churn Events Preview:


,churn_event_id,account_id,churn_date,reason_code,refund_amount_usd,preceding_upgrade_flag,preceding_downgrade_flag,is_reactivation,feedback_text
0,C-816288,A-c37cab,2024-10-27,pricing,4.03,0,0,0,switched to competitor
1,C-5a81e7,A-37f969,2024-06-25,support,96.45,1,0,0,NaN
2,C-a174be,A-b07346,2024-11-12,budget,0.00,0,0,0,missing features
3,C-accb39,A-1e50e0,2023-11-01,budget,54.94,0,0,0,switched to competitor
4,C-92f889,A-956988,2024-12-30,unknown,0.00,0,1,1,too expensive


## Export Final Cleaned Tables

After completing the data quality assessment and cleaning process, the final cleaned tables are exported as CSV files. These files will be used as the analytical input for the exploratory data analysis (EDA) stage and can also be reused in later Power BI modeling.
